# ChemBreak Colab Generator V3

V3 generates base harmful chemistry target tasks for later ChemBreak jailbreak evaluation.

Key V3 change: **Python controls the scenario assignment. Qwen does not choose scenario IDs.**

Generation is one candidate per model call, with immediate checkpointing to `candidate_tasks.csv`.

No paid LLM API is used.


In [ ]:
# 1. Confirm GPU runtime.
import torch, platform

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. Choose Runtime > Change runtime type > GPU."
    )

print("GPU:", torch.cuda.get_device_name(0))
print(
    "GPU memory (GB):",
    round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2)
)


In [ ]:
# 2. Clone the ChemBreak GitHub repository.

from pathlib import Path
import subprocess

REPO_URL = "https://github.com/Jollychuks/ChemBreak.git"
BRANCH = "main"
PROJECT_SUBDIR = "ChemBreak_Colab_Generator_v3"

CLONE_DIR = Path("/content/chembreak_repo")

if not CLONE_DIR.exists():
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, REPO_URL, str(CLONE_DIR)],
        check=True
    )
else:
    print("Repository already cloned:", CLONE_DIR)

PROJECT_DIR = (CLONE_DIR / PROJECT_SUBDIR).resolve()

if not PROJECT_DIR.exists():
    raise FileNotFoundError(
        f"PROJECT_DIR does not exist: {PROJECT_DIR}\n"
        "Confirm that the V3 folder has been uploaded to GitHub."
    )

print("PROJECT_DIR:", PROJECT_DIR)

print("\nFiles:")
for p in sorted(PROJECT_DIR.iterdir()):
    print(" -", p.name)


In [ ]:
# 3. Install Colab dependencies.

import subprocess, sys

subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "-r", str(PROJECT_DIR / "requirements_colab.txt")
    ],
    check=True
)

print("Dependencies installed.")


In [ ]:
# 4. Import V3 code and configuration.

import sys, json, importlib

sys.path.insert(0, str(PROJECT_DIR))

import generate_local
import validate_local

importlib.reload(generate_local)
importlib.reload(validate_local)

CONFIG_PATH = PROJECT_DIR / "run_config.json"
config = generate_local.load_config(CONFIG_PATH)

taxonomy = generate_local.load_taxonomy(
    PROJECT_DIR / config["taxonomy_file"]
)

print("Package version:", config["package_version"])
print("Generator prompt:", config["prompt_version"])
print("Validator prompt:", config["validator_prompt_version"])
print("Model:", config["model_id"])
print("Temperature:", config["temperature"])
print("Retries:", config["max_retries"])
print("Scenario control: PYTHON")
print("Scenario pair rate:", config["scenario_pair_rate"])
print("Pilot matrix IDs:", len(config["matrix_ids"]))
print("Candidates per row:", config["n_per_row"])


In [ ]:
# 5. Inspect the 18-row stratified pilot.

matrix = generate_local.load_matrix(
    PROJECT_DIR / config["matrix_file"]
)

selected = generate_local.select_rows(matrix, config)

print("Total matrix rows:", len(matrix))
print("Selected pilot rows:", len(selected))
print(
    "Expected raw candidates:",
    len(selected) * int(config["n_per_row"])
)

display(
    selected[
        [
            "MATRIX_ID", "HC_ID", "HC_CATEGORY",
            "HD_ID", "HAZARD_DOMAIN",
            "OT_ID", "OUTPUT_TYPE",
            "ALLOWED_SCENARIOS"
        ]
    ]
)


In [ ]:
# 6. Verify HC1-HC9 coverage.

coverage = (
    selected.groupby("HC_ID")["MATRIX_ID"]
    .count()
    .sort_index()
)

display(coverage.to_frame("matrix_rows"))

expected_hc = {f"HC{i}" for i in range(1, 10)}
actual_hc = set(selected["HC_ID"].astype(str))

if actual_hc != expected_hc:
    raise ValueError(
        f"Pilot does not cover HC1-HC9. Found: {sorted(actual_hc)}"
    )

print("Pilot covers HC1 through HC9.")


In [ ]:
# 7. Preview the Python-controlled scenario plan for the first row.

row = selected.iloc[0]

allowed = generate_local.split_scenarios(
    row["ALLOWED_SCENARIOS"]
)

row_seed = generate_local.stable_row_seed(
    int(config["seed"]),
    str(row["MATRIX_ID"])
)

scenario_plan = generate_local.build_scenario_plan(
    allowed,
    int(config["n_per_row"]),
    row_seed,
    pair_rate=float(config["scenario_pair_rate"])
)

print("Matrix ID:", row["MATRIX_ID"])
print("Allowed scenarios:", allowed)

for i, assigned in enumerate(scenario_plan, 1):
    print(
        f"C{i:04d}:",
        assigned if assigned else "NONE",
        "=>",
        generate_local.scenario_details(assigned, taxonomy)
    )


In [ ]:
# 8. Render one exact V3 prompt before loading the model.

template = generate_local.load_prompt_template(
    PROJECT_DIR / config["prompt_file"]
)

preview_prompt = generate_local.render_prompt(
    template,
    row,
    taxonomy,
    required_scenarios=scenario_plan[0],
    previous_prompts=[]
)

print(preview_prompt[:10000])
print("\n[Display truncated at 10,000 characters if needed]")


## Load the open-weight model

The default model is `Qwen/Qwen3-4B-Instruct-2507`.

It runs locally in the Colab GPU. No paid LLM API key is used.


In [ ]:
# 9. Load the generator model.

tokenizer, model = generate_local.load_local_model(
    config["model_id"],
    load_in_4bit=bool(config.get("load_in_4bit", True)),
    cache_dir=config.get("hf_cache_dir") or None
)


## V3 test

This test uses the same scenario-controller path as the full run.

Python assigns the scenario first. Qwen only writes the target task.


In [ ]:
# 10. Generate two V3 test candidates.

validated_test = generate_local.generate_test_candidates(
    row,
    n=2,
    template=template,
    taxonomy=taxonomy,
    tokenizer=tokenizer,
    model=model,
    config=config
)

print("\nValidated test candidates:", len(validated_test))

for i, item in enumerate(validated_test, 1):
    print(f"\n{'=' * 70}")
    print(f"Candidate {i}")
    print("=" * 70)
    print("\nBenchmark prompt:")
    print(item["benchmark_prompt"])
    print("\nMain goal:")
    print(item["main_goal"])
    print("\nChemical entity:")
    print(item["chemical_entity"])
    print("\nPython-controlled scenarios:")
    print(item["selected_scenarios"])
    print("\nDistinctive dimension:")
    print(item["distinctive_dimension"])


## Generate the 90-task stratified pilot

V3 generates one candidate per call and saves each accepted candidate immediately.


In [ ]:
# 11. Run the V3 90-task pilot.

output_path = generate_local.run_generation(
    PROJECT_DIR,
    config,
    tokenizer,
    model
)

print("\nGeneration output:", output_path)


In [ ]:
# 12. Review generated coverage.

import pandas as pd

generate_local.print_candidate_summary(output_path)

candidates = pd.read_csv(output_path)

display(
    candidates[
        [
            "candidate_id",
            "hc_id",
            "hd_id",
            "ot_id",
            "selected_scenarios",
            "benchmark_prompt",
            "prompt_version"
        ]
    ].tail(30)
)


## Optional semantic validation

This stage checks harmful intent, chemistry dependency, category fit, chemistry plausibility, scenario consistency, and jailbreak readiness.

It does not answer the harmful tasks.


In [ ]:
# 13. Run semantic validation.

validated_output_path = validate_local.run_validation(
    PROJECT_DIR,
    config,
    tokenizer,
    model
)

print("\nValidated output:", validated_output_path)


In [ ]:
# 14. Review semantic validation.

validate_local.print_validation_summary(
    validated_output_path
)

validated_df = pd.read_csv(validated_output_path)

display(
    validated_df[
        [
            "candidate_id",
            "hc_id",
            "hd_id",
            "ot_id",
            "validator_decision",
            "harmful_intent_score",
            "chemistry_dependency_score",
            "hc_fit_score",
            "hd_fit_score",
            "ot_fit_score",
            "chemistry_plausibility_score",
            "scenario_consistency_score",
            "jailbreak_readiness_score",
            "validator_reason"
        ]
    ].tail(30)
)


## Optional GitHub checkpoint

Generated CSV files are still in the temporary Colab clone until you push or download them.

Set `RUN_GITHUB_CHECKPOINT = True` only when you want to push.


In [ ]:
# 15. OPTIONAL: push generated outputs back to GitHub.

RUN_GITHUB_CHECKPOINT = False

if RUN_GITHUB_CHECKPOINT:
    from getpass import getpass
    import github_checkpoint

    token = getpass("GitHub token (input hidden): ")

    checkpoint_files = [
        PROJECT_DIR / "candidate_tasks.csv",
        PROJECT_DIR / "generation_progress.csv",
        PROJECT_DIR / "generation_errors.jsonl",
        PROJECT_DIR / "candidate_tasks_validated.csv",
        PROJECT_DIR / "validation_progress.csv",
        PROJECT_DIR / "validation_errors.jsonl",
    ]

    checkpoint_files = [
        p for p in checkpoint_files if p.exists()
    ]

    github_checkpoint.checkpoint_to_github(
        repo_dir=CLONE_DIR,
        files=checkpoint_files,
        commit_message="Checkpoint ChemBreak V3 generation",
        token=token,
        branch=BRANCH
    )


## Scaling later

The included `run_config_full.example.json` targets 6,775 raw candidates.

Do not switch to the full configuration until the V3 90-task pilot has been reviewed.
